# 03. Inventory Policy

This notebook converts predicted demand into simple inventory recommendations.

**Assumptions**
- Lead time: 7 days
- Service level: about 95% (Z = 1.65)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_csv("../data/retail_sales.csv")
df["date"] = pd.to_datetime(df["date"])

sample = df[(df["store_id"] == "store_1") & (df["item_id"] == "item_1")].copy()
sample = sample.sort_values("date").reset_index(drop=True)

sample["year"] = sample["date"].dt.year
sample["month"] = sample["date"].dt.month
sample["day"] = sample["date"].dt.day
sample["dayofweek"] = sample["date"].dt.dayofweek
sample["weekofyear"] = sample["date"].dt.isocalendar().week.astype(int)

train_size = int(len(sample) * 0.8)
train = sample.iloc[:train_size]
test = sample.iloc[train_size:]

features = ["year", "month", "day", "dayofweek", "weekofyear", "price", "promo"]

X_train = train[features]
y_train = train["sales"]
X_test = test[features]

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

results = test[["date", "sales"]].copy()
results["predicted_sales"] = rf_pred
results.head()

In [ ]:
lead_time_days = 7
z = 1.65  # about 95% service level

avg_daily_demand = results["predicted_sales"].mean()
std_daily_demand = results["predicted_sales"].std()

safety_stock = z * std_daily_demand * np.sqrt(lead_time_days)
reorder_point = avg_daily_demand * lead_time_days + safety_stock

print("Average predicted daily demand:", round(avg_daily_demand, 2))
print("Demand standard deviation:", round(std_daily_demand, 2))
print("Safety Stock:", round(safety_stock, 2))
print("Reorder Point:", round(reorder_point, 2))

## Interpretation
- Higher demand variability increases safety stock.
- Longer lead times increase the reorder point.
- Forecast-based inventory control is more realistic than using one fixed stock level for all products.